In [17]:
import csv
import shutil
from pathlib import Path
import json



## Save pathnames of all v12 ai-project-json files

In [18]:

def filter_json_to_csv(
    root_directory: str,
    target_field: str,
    target_value: str | int | float | bool,
    output_csv_path: str,
) -> int:
    """Recursively searches for 'ai_*.json' files where a specific field equals a target value

    and writes their file paths to a CSV.

    :param root_directory: The root folder path to start searching from.
    :param target_field: The key/field name inside the JSON object to check.
    :param target_value: The expected value for the target field.
    :param output_csv_path: Path where the output CSV file should be saved.
    :return: The total count of matching files found.
    """
    root_path = Path(root_directory)
    matching_files = []

    # `rglob` recursively searches all nested folders for matching filenames
    for file_path in root_path.rglob("ai_*.json"):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

                # Ensure the JSON content is a dictionary before checking keys
                if (
                    isinstance(data, dict)
                    and data.get(target_field) == target_value
                ):
                    matching_files.append([str(file_path.resolve())])

        except (json.JSONDecodeError, OSError) as e:
            # Safely skip corrupted JSONs or unreadable files
            print(f"Skipping file due to error ({file_path}): {e}")

    # Write matches to CSV
    with open(
        output_csv_path, "w", newline="", encoding="utf-8"
    ) as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["file_path"])  # Header column
        writer.writerows (matching_files)

    print(
        f"Done! Found {len(matching_files)} matching files out of all checked."
    )
    return len(matching_files)

In [19]:
proj_dump_dir = '/home/hemduttdabral/Downloads/proj-api-volume-dump'

filter_json_to_csv(
    root_directory=proj_dump_dir,
    target_field="schema_version",
    target_value="v12",
    output_csv_path="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_projs.csv")

Skipping file due to error (/home/hemduttdabral/Downloads/proj-api-volume-dump/64b124bd-241c-4cca-b79e-474d40d8de62/workspace/projects/new_project_name_700039/ai_new_project_name_700039.json): Expecting value: line 1 column 1 (char 0)
Skipping file due to error (/home/hemduttdabral/Downloads/proj-api-volume-dump/64b124bd-241c-4cca-b79e-474d40d8de62/workspace/projects/new_project_name_164926/ai_new_project_name_164926.json): Expecting value: line 1 column 1 (char 0)
Skipping file due to error (/home/hemduttdabral/Downloads/proj-api-volume-dump/64b124bd-241c-4cca-b79e-474d40d8de62/workspace/projects/new_project_name_449184/ai_new_project_name_449184.json): Expecting value: line 1 column 1 (char 0)
Skipping file due to error (/home/hemduttdabral/Downloads/proj-api-volume-dump/64b124bd-241c-4cca-b79e-474d40d8de62/workspace/projects/new_project_name_467444/ai_new_project_name_467444.json): Expecting value: line 1 column 1 (char 0)
Skipping file due to error (/home/hemduttdabral/Downloads/pr

111

In [20]:
## copy all v12 files to a new folder

In [21]:
def copy_and_rename_from_csv(
    csv_file_path: str, destination_dir: str, start_index: int = 1
) -> int:
    """Reads file paths from a CSV, copies them to a destination directory,

    and renames them using the pattern 'ai_v12_{i}.json'.

    :param csv_file_path: Path to the CSV file generated previously.
    :param destination_dir: Directory where copied files will be placed.
    :param start_index: Starting serial number for 'i' (defaults to 1).
    :return: Total number of files successfully copied.
    """
    dest_path = Path(destination_dir)
    dest_path.mkdir(
        parents=True, exist_ok=True
    )  # Ensures target directory exists

    copied_count = 0

    with open(csv_file_path, "r", encoding="utf-8") as csv_file:
        reader = csv.reader(csv_file)

        # Skip header if present ('file_path')
        header = next(reader, None)
        if header and header[0] != "file_path":
            # If the first row wasn't the header, reset file pointer
            csv_file.seek(0)

        for i, row in enumerate(reader, start=start_index):
            if not row:
                continue

            src_file_path = Path(row[0].strip())

            if not src_file_path.exists():
                print(f"Skipping (File not found): {src_file_path}")
                continue

            # Keep original extension (.json)
            extension = src_file_path.suffix or ".json"
            new_filename = f"ai_v12_{i}{extension}"
            target_path = dest_path / new_filename

            try:
                shutil.copy2(src_file_path, target_path)
                copied_count += 1
            except OSError as e:
                print(f"Failed to copy {src_file_path}: {e}")

    print(
        f"Successfully copied and renamed {copied_count} files into '{destination_dir}'."
    )
    return copied_count

In [ ]:
copy_and_rename_from_csv(
    csv_file_path="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_projs.csv",
    destination_dir="/home/hemduttdabral/projects/experimental/staiotcraft-json-analysis/v12_projs_copied",
    start_index=0)